In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

(a) 

In [4]:
D = pd.read_csv("lines.csv", delimiter=",").values
X_cols = D[:, :3]  
Y_cols = D[:, 3:]  

x = X_cols[:, 0]
y = Y_cols[:, 0]

x_mean, y_mean = np.mean(x), np.mean(y)
A = np.column_stack([x - x_mean, y - y_mean])

U, S, Vt = np.linalg.svd(A)
a, b = Vt[-1, :]  


d = a * x_mean + b * y_mean
print(f"(a) Line 1 (TLS): {a:.6f}x + {b:.6f}y = {d:.6f}")

(a) Line 1 (TLS): 0.773562x + -0.633721y = 3.794192


(b)

In [5]:
def ransac_line(x, y, thresh=0.5, max_iter=500):
    best = (0, None, None, None, None)
    for _ in range(max_iter):
        idx = np.random.choice(len(x), 2, replace=False)
        a0 = y[idx[1]] - y[idx[0]]
        b0 = -(x[idx[1]] - x[idx[0]])
        d0 = (x[idx[1]] - x[idx[0]]) * y[idx[0]] - (y[idx[1]] - y[idx[0]]) * x[idx[0]]
        norm = np.sqrt(a0**2 + b0**2)
        a0, b0, d0 = a0/norm, b0/norm, d0/norm
        inliers = np.abs(a0*x + b0*y - d0) < thresh
        if np.sum(inliers) > best[0]:
            x_in, y_in = x[inliers], y[inliers]
            A_in = np.column_stack([x_in - np.mean(x_in), y_in - np.mean(y_in)])
            U, S, Vt = np.linalg.svd(A_in)
            a1, b1 = Vt[-1, :]
            d1 = a1 * np.mean(x_in) + b1 * np.mean(y_in)
            best = (np.sum(inliers), a1, b1, d1, inliers)
    return best[1], best[2], best[3], best[4]

In [6]:
X_all, Y_all = X_cols.flatten(), Y_cols.flatten()
remaining = np.ones(len(X_all), dtype=bool)

print("\nRANSAC results:")
for i in range(3):
    a, b, d, inliers = ransac_line(X_all[remaining], Y_all[remaining])
    idx = np.where(remaining)[0][inliers]
    print(f"Line {i+1}: {a:.6f}x + {b:.6f}y = {d:.6f} ({len(idx)} points)")
    remaining[idx] = False


RANSAC results:
Line 1: 0.718235x + -0.695801y = -0.363991 (65 points)
Line 2: -0.503269x + -0.864130y = -1.732244 (59 points)
Line 3: 0.912462x + -0.409162y = 2.941589 (37 points)
